# PDF Text Extractor for Text-to-Speech (TTS)

This notebook allows you to upload a PDF file, extract its text content, and clean it up. The resulting text is stored in a variable, ready to be used with a Text-to-Speech (TTS) engine.

**Steps:**
1.  Run the cells sequentially.
2.  Upload your PDF file using the widget.
3.  The notebook will extract and clean the text.
4.  The final text will be available in the `processed_text` variable.

## Prerequisites and Installation

This notebook requires the following Python libraries:

*   `PyPDF2`: For reading and extracting text from PDF files.
*   `ipywidgets`: For the interactive file upload widget in Jupyter.
*   `notebook`: The Jupyter Notebook environment itself.
*   `google-genai`: For using Google's generative AI models (e.g., for summarization or TTS, though TTS itself might be a separate library).

You can install these libraries using pip. Run the following command in your terminal or a code cell in Jupyter (uncomment it first):

```python
# !pip install PyPDF2 ipywidgets notebook google-genai
```

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

upload_widget = FileUpload(accept='.pdf', multiple=False)
display(upload_widget)

The following code will extract text from the uploaded PDF using the `PyPDF2` library. It includes error handling for potential issues during the extraction process, such as encrypted or corrupted PDF files.

In [ ]:
import io
from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError

def extract_text_from_pdf(upload_value):
    """Extracts text from an uploaded PDF file."""
    extracted_text = ""
    if not upload_value: # Check if the upload_value is empty
        print("No file uploaded.")
        return extracted_text

    # Get the content of the first uploaded file
    # The structure of upload_value is a list of dictionaries,
    # each dictionary representing a file.
    # We're interested in the 'content' of the first file.
    first_file_info = upload_value[0]
    file_content = first_file_info['content']

    try:
        pdf_file = io.BytesIO(file_content)
        pdf_reader = PdfReader(pdf_file)
        
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            extracted_text += page.extract_text() + "\n"
            
        print("Text extracted successfully.")
        # Print the first 500 characters of the extracted text as a sample
        print("Sample of extracted text (first 500 chars):\n", extracted_text[:500])
        
    except PdfReadError:
        print("Error: Could not read PDF. It might be encrypted or corrupted.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        
    return extracted_text

# --- Example usage (illustrative, run this cell after uploading a PDF) ---
# # Make sure you have run the cell above where 'upload_widget' is defined and you have uploaded a file.
# extracted_text = "" # Initialize to prevent potential NameError if upload fails
# if upload_widget.value:
#     extracted_text = extract_text_from_pdf(upload_widget.value)
# else:
#     print("Please upload a PDF file first using the widget above.")


The following Python code will perform basic cleaning on the extracted text. This includes removing extra spaces and normalizing newlines to prepare the text for potential Text-to-Speech (TTS) application or other text processing tasks.

In [ ]:
import re

def clean_text(raw_text):
    """Cleans the raw text by normalizing whitespace and paragraph breaks."""
    if not isinstance(raw_text, str):
        print("Input is not a string. Returning as is.")
        return raw_text

    text = raw_text
    # 1. Replace Windows-style newlines and old Mac-style newlines with standard newlines
    text = text.replace('\r\n', '\n')
    text = text.replace('\r', '\n')

    # 2. Replace multiple newlines (more than 2) with a unique placeholder
    text = re.sub(r'\n{2,}', '__PARAGRAPH_BREAK__', text)

    # 3. Replace remaining single newlines with a space (to join lines within a paragraph)
    text = text.replace('\n', ' ')

    # 4. Replace the placeholder with double newlines (to restore paragraph breaks)
    text = text.replace('__PARAGRAPH_BREAK__', '\n\n')

    # 5. Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # 6. Strip leading/trailing whitespace from the whole text
    cleaned_text = text.strip()

    return cleaned_text

# --- Example usage (illustrative) ---
# # Assuming 'extracted_text' variable holds the text from the PDF extraction cell.
# # If that cell hasn't been run or if no file was uploaded, extracted_text might be empty or not defined.
# # For demonstration, let's use a sample raw text. 
# # In a real run, you would use: processed_text = clean_text(extracted_text)

# # Initialize extracted_text for safety, in case the previous cell was not run
# extracted_text_sample = "This   is a test.\n\nThis is another line with   many spaces.\nAnd one more.\r\nThis should be a new paragraph.
\n\nThis is a third paragraph after too many newlines."
# print(f"Original sample text:\n'{extracted_text_sample}'\n")

# processed_text_sample = clean_text(extracted_text_sample)
# print(f"Cleaned sample text:\n'{processed_text_sample}'\n")

# # Actual call using the variable from the previous cell (uncomment to use)
# # Ensure 'extracted_text' is defined and contains the PDF text before running this.
# # For example, you might want to re-run the extraction cell, then this cell.
# processed_text = "" # Initialize to prevent NameError
# if 'extracted_text' in locals() and extracted_text:
#     processed_text = clean_text(extracted_text)
#     print("Cleaned extracted PDF text (first 500 chars):\n", processed_text[:500])
# elif 'extracted_text' in locals():
#     print("Extracted text is empty. Cleaning resulted in empty text.")
# else:
#     print("Variable 'extracted_text' not found. Run the PDF extraction cell first.")


## Text-to-Speech using Google GenAI

The following section allows you to convert the cleaned text (from the `processed_text` variable above) into an audio file using Google's Generative AI.

You will need a Google API Key for the Gemini models (which includes TTS capabilities). Please have your key ready.

In [ ]:
# @title Enter your Google API Key
# This cell prompts you to enter your Google API Key.
# The key is stored in the 'api_key' variable and will be used by the next cell to authenticate with the Google GenAI service.

import os # Used for illustration, but input() is simpler for direct notebook use.

print("Please enter your Google API Key. This key will be used to authenticate with the Google GenAI service for Text-to-Speech.")
api_key = input("Enter your Google API Key: ")

if api_key:
    print("API Key stored. You can now proceed to the next cell to generate audio.")
else:
    print("No API Key entered. Please run this cell again and provide your key if you wish to use the TTS feature.")

In [ ]:
# @title Generate Audio from Text using Google GenAI TTS
# This cell uses the API key you provided and the 'processed_text' (from PDF extraction)
# to generate an audio file.

import os
import wave # Make sure wave is imported
try:
    import google.genai as genai
    from google.genai import types
except ImportError:
    print("ERROR: google-genai library not found. Please install it using: !pip install google-genai")
    # You might want to raise an error or handle this more gracefully
    google_genai_available = False
else:
    google_genai_available = True

# Function to save PCM audio data to a WAV file
def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
   with wave.open(filename, "wb") as wf:
      wf.setnchannels(channels)
      wf.setsampwidth(sample_width)
      wf.setframerate(rate)
      wf.writeframes(pcm)

if google_genai_available and 'api_key' in locals() and api_key:
    # Ensure 'processed_text' is defined by running the previous cells
    # First, run the PDF extraction and cleaning cells to define 'extracted_text' and 'processed_text'
    if 'upload_widget' in locals() and upload_widget.value: # Check if a file was uploaded
      if 'extracted_text' not in locals() or not extracted_text: # If not already extracted
          extracted_text = extract_text_from_pdf(upload_widget.value)
      if extracted_text: # If extraction was successful
          if 'processed_text' not in locals() or not processed_text: # If not already cleaned
              processed_text = clean_text(extracted_text)
      else:
          print("Error: Text extraction failed or PDF was empty. Cannot proceed with TTS.")
          processed_text = "" # Ensure it's defined as empty to avoid NameError below
    else:
        print("Error: No PDF file uploaded. Please upload a file and run previous cells first.")
        processed_text = "" # Ensure it's defined as empty
        extracted_text = "" # Ensure it's defined as empty

    if 'processed_text' in locals() and processed_text:
        print("\nStarting Text-to-Speech conversion...")
        
        output_filename_prompt = "Enter the desired output filename (e.g., audio.wav) [default: output.wav]: "
        output_filename = input(output_filename_prompt).strip()
        if not output_filename:
            output_filename = "output.wav"
        if not output_filename.lower().endswith(".wav"):
            output_filename += ".wav"

        try:
            print(f"Initializing Google GenAI client...")
            client = genai.Client(api_key=api_key)

            print(f"Requesting audio generation for the processed text...")
            # Ensure processed_text is not empty or just whitespace
            if not processed_text.strip():
                print("Error: The 'processed_text' is empty. Cannot generate audio.")
            else:
                response = client.models.generate_content(
                   model="gemini-2.5-flash-preview-tts", # Using the model from user's code
                   contents=[processed_text], # Pass the actual processed text
                   config=types.GenerateContentConfig(
                      response_modalities=["AUDIO"],
                      speech_config=types.SpeechConfig(
                         voice_config=types.VoiceConfig(
                            prebuilt_voice_config=types.PrebuiltVoiceConfig(
                               voice_name='Kore', # Using the voice from user's code
                            )
                         )
                      ),
                   )
                )

                print("Audio data received. Saving to file...")
                audio_data = response.candidates[0].content.parts[0].inline_data.data
                
                wave_file(output_filename, audio_data)
                print(f"SUCCESS: Audio content successfully saved to '{output_filename}'")
                print("You should be able to find this file in the same directory as this notebook.")

        except ImportError: # Should be caught by the top-level check, but good for safety
            print("ERROR: google-genai library is not installed. Please run the prerequisite cell to install it.")
        except NameError as ne:
            if 'api_key' in str(ne):
                 print("ERROR: API Key not found. Please run the previous cell to enter your API key.")
            elif 'processed_text' in str(ne):
                print("ERROR: 'processed_text' not found. Please ensure the PDF processing cells have been run successfully.")
            else:
                print(f"An unexpected NameError occurred: {ne}. Please check cell execution order.")
        except Exception as e:
            print(f"An error occurred during TTS processing: {e}")
            print("Please check the following:")
            print("1. Your Google API Key is correct and has the necessary permissions for the GenAI TTS service.")
            print("2. You have a stable internet connection.")
            print("3. The 'processed_text' variable contains valid text from the PDF.")
            print("4. The service model 'gemini-2.5-flash-preview-tts' is available for your key.")

    else:
        print("ERROR: 'processed_text' not found or is empty. Please run the PDF extraction and cleaning cells before running TTS.")
elif not google_genai_available:
    print("Skipping TTS: google-genai library is not available. Please install it.")
else:
    print("Skipping TTS: API Key not provided. Please run the previous cell to enter your API key.")


## How to Use This Notebook and Final Outputs

This notebook guides you through extracting text from a PDF and then converting that text into an audio file using Google's Text-to-Speech service.

**Please follow these steps carefully:**

1.  **Check Prerequisites**: Ensure you have installed all necessary libraries mentioned in the "Prerequisites and Installation" section at the beginning of this notebook.
2.  **Run Cells Sequentially**: Execute each code cell in this notebook from top to bottom. This is important for the variables and functions to be defined correctly.
3.  **Upload PDF**:
    *   Run the cell that displays the "Upload PDF" button.
    *   Click the button and select your PDF file.
4.  **Text Extraction and Cleaning**:
    *   The cells following the upload will automatically extract text from your PDF and clean it.
    *   The cleaned text will be stored in a Python variable named `processed_text`. A sample of this text is usually printed.
5.  **Provide Google API Key for TTS**:
    *   Run the cell that asks for your "Google API Key".
    *   Enter your valid API key when prompted. This key is required to use the Google Text-to-Speech service.
6.  **Generate Audio File (TTS)**:
    *   Run the cell titled "Generate Audio from Text using Google GenAI TTS".
    *   This cell will use the `processed_text` and your API key.
    *   You will be prompted to enter a filename for your audio output (e.g., `my_audio.wav`). If you just press Enter, it will default to `output.wav`.
    *   If successful, an audio file will be generated.
7.  **Access Your Outputs**:
    *   **Cleaned Text**: The variable `processed_text` holds the cleaned text extracted from the PDF. You can use this variable directly in other Python scripts or cells if needed.
    *   **Audio File**: The generated `.wav` audio file (e.g., `output.wav` or the name you provided) will be saved in the **same directory where this Jupyter Notebook (`pdf_to_text.ipynb`) is located**. Check your file system in that location to find it.

If you encounter any errors, please read the error messages carefully. They often provide clues about missing installations, incorrect API keys, or issues with the input PDF.